# CFU prediction using TPM values from bulk RNA-seq

## Data loading

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np

Load TPM data and CFU counts.

In [2]:
from src.tpm_data import get_all_tpm_data

# Paths to Fcnts and CFU data
fcnts_path = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/fcnts_timezero"
cfu_path   = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/all_cfus"

# Get data
data_df = get_all_tpm_data(
    fcnts_path = fcnts_path,
    cfu_path = cfu_path
)

In [3]:
data_df

,SP_0001,SP_0002,SP_0003,SP_0004,SP_0005,SP_0006,SP_0007,SP_0008,SP_0009,SP_0010,...,SP_2239,SP_2240,CFU,drug_id,num_drugs,drug1,drug2,drug1_dose,drug2_dose,timepoint
Labels,,,,,,,,,,,,,,,,,,,,,
NDC1hr-a,91.887204,73.003181,59.434830,165.474181,28.785771,24.931266,31.328863,102.009834,13.078184,73.268747,...,1232.456175,1609.040485,580000000,NDC,1,NDC,NaN,NaN,NaN,1
NDC1hr-b,94.097834,80.591203,64.648788,175.865826,33.672335,23.812941,32.464085,88.927461,45.302669,86.844990,...,1234.082514,1664.082222,1000000000,NDC,1,NDC,NaN,NaN,NaN,1
NDC1hr-c,92.563834,72.890042,44.378034,158.424808,30.841672,22.895893,18.811935,85.754892,24.501398,75.203266,...,1369.580457,1827.791359,800000000,NDC,1,NDC,NaN,NaN,NaN,1
14CEF1hr-a,64.432483,52.754189,54.063293,128.259588,28.998239,21.943509,47.865737,64.651174,9.697676,49.818086,...,1958.037472,1537.434864,340000000,CEF,1,CEF,NaN,0.25,NaN,1
14CEF1hr-b,67.824743,63.344749,35.632113,127.202811,24.721847,22.006278,32.252596,65.061409,10.607838,45.240048,...,1769.710527,1570.874119,700000000,CEF,1,CEF,NaN,0.25,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34CEF12RIF4hr-b,250.462160,330.726238,38.232498,255.911075,16.701565,23.053869,11.885009,98.896800,12.899583,122.530787,...,962.459468,3231.829820,30000000,CEF+RIF,2,CEF,RIF,0.75,0.50,4
34CEF12RIF4hr-c,226.255231,240.873028,62.899433,230.612976,24.506621,27.978551,31.707580,115.861558,20.648594,98.402187,...,946.507463,2893.918199,32000000,CEF+RIF,2,CEF,RIF,0.75,0.50,4
34CEF34RIF4hr-a,237.790268,299.804077,38.604780,250.452101,36.971501,33.600879,26.309308,140.271276,9.017439,100.513536,...,966.513847,3036.631548,8000000,CEF+RIF,2,CEF,RIF,0.75,0.75,4


## Pre-processing

Check for CFU = 0, which indicates fully dead cells.

In [4]:
# Find idx where CFU = 0, then list the sample ID
zero_idx = np.where(data_df["CFU"] == 0)[0]
print(data_df.index[zero_idx].tolist())

# Check CFU values for the other 2 replicates
rep_names = ["34CEF4hr-a", "34CEF4hr-b"]
cfu_val_check = data_df.loc[rep_names]["CFU"].tolist()
print(f"CFU values for 34CEF4hr-a and 34CEF4hr-b :{cfu_val_check}")

# Remove sample
data_df = data_df[data_df["CFU"] != 0]

['34CEF4hr-c']
CFU values for 34CEF4hr-a and 34CEF4hr-b :[20000000, 20000000]


## Training with stratified split

PLS regression CV scheme.

In [ ]:
from src.split import combination_stratified_split
from sklearn.linear_model import ElasticNetCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import GridSearchCV

# Define a function to run nested EN CV
def run_nested_pls_cv(df, splits):

    scores = []

    for train_idx, test_idx in splits:
        train_df = df.iloc[train_idx]
        X_train = train_df.iloc[:, train_df.columns.str.contains("SP")]
        y_train = train_df["CFU"]

        test_df = df.iloc[test_idx]
        X_test = test_df.iloc[:, test_df.columns.str.contains("SP")]
        y_test = test_df["CFU"]

        # Create a nested hyperparameter tuning scheme
        param_grid = {
            "model__n_components": list(range(3, 20))
        }

        # Make pipeline for PLS regression
        pipeline = Pipeline([
                ("scaler", StandardScaler()),
                ("model", PLSRegression())
        ])

        # Setup GridSearch
        search = GridSearchCV(
            estimator = pipeline,
            cv = 5,
            param_grid = param_grid,
            scoring = "neg_mean_squared_error",
        )

        # Fit with best params
        search.fit(X_train, y_train)
        preds = search.predict(X_test)

        # Evaluate
        score = r2_score(y_test, preds)
        scores.append(score)
    
    mean_score = np.mean(scores)

    # Round
    scores = [round(score, 3) for score in scores]
    mean_score = round(mean_score, 3)

    return scores, mean_score
    
# Make stratified splits
strat_splits = combination_stratified_split(data_df, num_folds = 5, seed = 111)

# Run Nested CV
scores1, mean_score1 = run_nested_pls_cv(
    df = data_df,
    splits = strat_splits
)

print(f"R^2 for 5 folds : {scores1}")
print(f"Mean R^2 : {mean_score1}")

R^2 for 5 folds : [0.856, 0.847, 0.759, 0.859, 0.807]
Mean R^2 : 0.825


: 

## Training with held-out combination(s)

All single cell data.

In [ ]:
# Get all single data
single_df = data_df.iloc[data_df["nu"]]

## Training with sparse combination matrix